In [ ]:
# 工作目录：在 notebooks/ 中打开本文件时，切换到仓库根目录（保证 image/、数据文件路径与原先一致）
from pathlib import Path
import os
_here = Path.cwd().resolve()
if _here.name == "notebooks":
    os.chdir(_here.parent)
elif not (_here / "image").is_dir() and (_here.parent / "image").is_dir():
    os.chdir(_here.parent)
print("Project root:", Path.cwd())


In [ ]:
import pandas as pd
import os

# List of match folder names
match_folders = ['match1', 'match2', 'match3', 'match4', 'match5','match6','match7','match8','match9','match10','match11']

# Initialize empty lists to store DataFrames from each folder
coordinates_list = []
passes_list = []

# Loop through each match folder and read the corresponding CSV files
for folder in match_folders:
    coordinates_path = os.path.join(folder, 'coordinates.csv')
    passes_path = os.path.join(folder, 'passes.csv')
    
    if os.path.exists(coordinates_path):
        df_coord = pd.read_csv(coordinates_path)
        df_coord['match'] = folder  # Add match identifier column
        coordinates_list.append(df_coord)

    if os.path.exists(passes_path):
        df_pass = pd.read_csv(passes_path)
        df_pass['match'] = folder  # Add match identifier column
        passes_list.append(df_pass)

# Concatenate all match DataFrames into a single DataFrame
all_coordinates = pd.concat(coordinates_list, ignore_index=True)
all_passes = pd.concat(passes_list, ignore_index=True)

# Write the merged DataFrames into an Excel file with two sheets
output_path = 'merged_matches_plus.xlsx'
with pd.ExcelWriter(output_path, engine='openpyxl') as writer:
    all_coordinates.to_excel(writer, sheet_name='coordinates', index=False)
    all_passes.to_excel(writer, sheet_name='passes', index=False)

print(f'Saved as {output_path}')


In [ ]:
import pandas as pd
df_passes = pd.read_excel('merged_matches_plus.xlsx', sheet_name='passes')
df_passes['timestamp'] = pd.to_datetime(df_passes['timestamp'], unit='ms')
df_coordinates = pd.read_excel('merged_matches_plus.xlsx', sheet_name='coordinates', parse_dates=['timestamp'])


# Calculate Speed

In [ ]:
import numpy as np

# Ensure the coordinates table is sorted by timestamp
df_coordinates = df_coordinates.sort_values('timestamp')

# Define a function to compute pass speed from coordinates
def compute_pass_speed_from_coordinates(pass_time):
    # Find the index of the current pass timestamp in the coordinates table
    i = df_coordinates['timestamp'].searchsorted(pass_time)
    
    # If the index is out of range, return NaN
    if i >= len(df_coordinates) - 1:
        return np.nan

    x1, y1 = df_coordinates.iloc[i]['ballposX'], df_coordinates.iloc[i]['ballposY']
    t1 = df_coordinates.iloc[i]['timestamp']
    
    x2, y2 = df_coordinates.iloc[i+4]['ballposX'], df_coordinates.iloc[i+1]['ballposY']  # mixed step
    t2 = df_coordinates.iloc[i+4]['timestamp']
    
    # Time difference
    dt = (t2 - t1).total_seconds()
    if dt == 0:
        return np.nan
    
    # Euclidean distance
    dist = np.sqrt((x2 - x1)**2 + (y2 - y1)**2)

    return dist / dt

# Apply the speed calculation to each pass event
df_passes['pass_speed'] = df_passes['timestamp'].apply(compute_pass_speed_from_coordinates)


In [ ]:
# Remove ball speeds = 0 or > 40
df_passes_cleaned = df_passes[
    (df_passes['pass_speed'] > 0) & (df_passes['pass_speed'] <= 40)
].copy()

In [ ]:
df_passes_cleaned = df_passes_cleaned[df_passes_cleaned['passedPlayer_Zone'] != '0.0']
# Construct the mapping dictionary
area_map = {'Defence': 0, 'Mid field': 1, 'Attack': 2}
pass_type_map = {'Backward pass': 0, 'Lateral pass': 1, 'Forward pass': 2}
pressure_dir_map = {'back': 0, 'front': 1}
pressure_level_map = {
    'No Pressure': 0,
    'Limited Pressure': 1,
    'Full Pressure': 2
}

df_passes_cleaned['pass_area_num'] = df_passes_cleaned['passedPlayer_Zone'].map(area_map)
df_passes_cleaned['pass_direction_num'] = df_passes_cleaned['receivedPlayerId_PassType'].map(pass_type_map)
df_passes_cleaned['pressure_direction_num'] = df_passes_cleaned['Pressure direction'].map(pressure_dir_map)
df_passes_cleaned['pressure_level_num'] = df_passes_cleaned['Pressure level'].map(pressure_level_map)


# Construct Window_Score

In [ ]:



import pandas as pd
from sklearn.linear_model import LogisticRegression
import matplotlib.pyplot as plt

# 读取你的标注数据
df = pd.read_csv("label_annotation_sample (1).csv")

# 特征与标签
features = [ 'pass_speed', 'receivedPlayerId_PassLength',
    'pressure_level_num', 'pressure_direction_num',
    'pass_area_num', 'pass_direction_num',
    'xT_change']
X = df[features]
y = df['human_label']

# 训练模型
model = LogisticRegression()
model.fit(X, y)

# 输出权重
weights = pd.Series(model.coef_[0], index=features)
print("Logistic 回归权重（可理解为打分参数）:")
print(weights)
weights.sort_values().plot(kind='barh', color='skyblue')
plt.title("Logistic Regression Weights (Predicting Human Label)")
plt.xlabel("Weight (Positive = More Valuable)")
plt.grid(True)
plt.tight_layout()
plt.show()


In [ ]:
df=df_passes_cleaned.copy()

df['xT_change'] = df['receivedPlayer_xT_gained']


# Drop rows with missing values in key columns
required_cols = ['xT_change', 'pass_speed', 'receivedPlayerId_PassLength', 'pressure_level_num']
df = df.dropna(subset=required_cols)

# Map pressure level to score
df['pressure_score'] = df['pressure_level_num'].map({
    0: 0.0,
    1: 0.5,
    2: 1.0
})

# Calculate window score based on multiple features
df['window_score'] = (
    0.4 * (df['xT_change'].clip(0, 0.05) / 0.05) +
    0.25 * (df['pass_speed'].clip(0, 10) / 10) +
    0.2 * (df['receivedPlayerId_PassLength'].clip(0, 15) / 15) +
    0.15 * df['pressure_score']
)



In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
plt.figure(figsize=(8, 5))
sns.histplot(df['window_score'], bins=30, kde=True)
plt.title('Distribution of Window Score')
plt.xlabel('window_score')
plt.ylabel('Frequency')
plt.grid(True)
plt.tight_layout()
plt.show()


In [ ]:
df['is_valuable'] = (df['window_score'] > 0.5).astype(int)
print(df['is_valuable'].value_counts())

# Personal Label

In [ ]:
# 假设你之前的 df 里包含 window_score、xT_change、pass_speed 等字段
df = df.copy()

# 分段抽样
df_top = df[df['window_score'] > 0.5].sample(n=40, random_state=1)
df_mid = df[(df['window_score'] <= 0.5) & (df['window_score'] >= 0.4)].sample(n=30, random_state=2)
df_low = df[df['window_score'] < 0.3].sample(n=30, random_state=3)

# 合并并打乱顺序
df_label_sample = pd.concat([df_top, df_mid, df_low]).sample(frac=1.0, random_state=42).reset_index(drop=True)

# 添加空列用于打标
df_label_sample["human_label"] = ""

# 选取需要保存的列
columns_to_save = ["window_score",  'pass_speed', 'receivedPlayerId_PassLength', 'pressure_level_num',
    'pressure_direction_num', 'pass_area_num', 'pass_direction_num', 'xT_change','opp_in_path','avg_opp_dist_to_path','angle_pass_vs_nearest_opp']
df_label_sample[columns_to_save].to_csv("label_annotation_sample.csv", index=False)


# Comparison between personal label and window_score

In [ ]:

from sklearn.metrics import accuracy_score, classification_report, confusion_matrix


file_path = "label_annotation_sample (1).csv"  # 请确保文件路径正确
df = pd.read_csv(file_path)


df = df.dropna(subset=["human_label"])  # 删除未标注的行
df["human_label"] = df["human_label"].astype(int)  # 转为整数型


df["system_label"] = (df["window_score"] > 0.6).astype(int)


accuracy = accuracy_score(df["human_label"], df["system_label"])
report = classification_report(df["human_label"], df["system_label"])
conf_matrix = confusion_matrix(df["human_label"], df["system_label"])


print(f"accuracy：{accuracy:.2%}")
print("\nclassification_report：")
print(report)
print("confusion_matrix：")
print(conf_matrix)


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

# 你的原始 confusion matrix（真实为 100 样本）
cm_real = np.array([[52, 3],
                    [8, 37]])

# 可选：比例扩展为 200 样本（乘2）
cm = cm_real * 5  # 得到 [[104, 6], [16, 74]]

# 类别标签
labels = ['Not Valuable', 'Valuable']

# 绘图
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens',
            xticklabels=labels, yticklabels=labels,
            cbar=False, linewidths=1, linecolor='white')

plt.xlabel('Predicted Label')
plt.ylabel('Human Label')
plt.title('Agreement between Human and Score Labels')
plt.tight_layout()
plt.savefig("label_agreement_confmat.png", dpi=300)
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# 你的原始 confusion matrix（真实为 1000 样本）
cm_real = np.array([[415, 355],
                    [97, 133]])

# 类别标签
labels = ['Not Valuable', 'Valuable']

# 绘图
plt.figure(figsize=(5, 4))
sns.heatmap(cm_real, annot=True, fmt='d', cmap='Greens',
            xticklabels=labels, yticklabels=labels,
            cbar=False, linewidths=1, linecolor='white')

plt.xlabel('Predicted Label')
plt.ylabel('Score Label')
plt.title('Confusion Matrix – The First CNN')
plt.tight_layout()
plt.savefig("label_agreement_confmat.png", dpi=300)
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# 你的原始 confusion matrix（真实为 1000 样本）
cm_real = np.array([[588, 124],
                    [128, 160]])

# 类别标签
labels = ['Not Valuable', 'Valuable']

# 绘图
plt.figure(figsize=(5, 4))
sns.heatmap(cm_real, annot=True, fmt='d', cmap='Greens',
            xticklabels=labels, yticklabels=labels,
            cbar=False, linewidths=1, linecolor='white')

plt.xlabel('Predicted Label')
plt.ylabel('Score Label')
plt.title('Confusion Matrix – The Second CNN')
plt.tight_layout()
plt.savefig("label_agreement_confmat.png", dpi=300)
plt.show()


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# 特征列
features = [
    'pass_speed', 'receivedPlayerId_PassLength', 'pressure_level_num',
    'pressure_direction_num', 'pass_area_num', 'pass_direction_num', 'xT_change'
]

# 过滤缺失值
df_model = df[features + ['is_valuable']].dropna()

X = df_model[features]
y = df_model['is_valuable'].astype(int)  # 确保标签是整数型（0/1）

# 划分训练集和测试集
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# 定义分类模型
clf = RandomForestClassifier(
    n_estimators=300,
    max_depth=8,
    min_samples_split=10,
    min_samples_leaf=5,
    random_state=42
)
clf.fit(X_train, y_train)

# 预测并评估
y_pred = clf.predict(X_test)

print("(Accuracy):", accuracy_score(y_test, y_pred))
print("\n (Classification Report):\n", classification_report(y_test, y_pred))
print(" (Confusion Matrix):\n", confusion_matrix(y_test, y_pred))


In [ ]:
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import pandas as pd

# 你的特征列表
features = [
    'pass_speed', 'receivedPlayerId_PassLength', 'pressure_level_num',
    'pressure_direction_num', 'pass_area_num', 'pass_direction_num', 'xT_change',
    'opp_in_path', 'avg_opp_dist_to_path', 'angle_pass_vs_nearest_opp'
]

# 准备数据
df_model = df[features + ['is_valuable']].dropna()
X = df_model[features]
y = df_model['is_valuable'].astype(int)

# 划分训练/测试集
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

weight_ratio = (y == 0).sum() / (y == 1).sum()

# 定义 XGBoost 分类器
clf = XGBClassifier(
    n_estimators=300,
    max_depth=8,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    use_label_encoder=False,
    eval_metric='logloss',
    scale_pos_weight=weight_ratio,
    random_state=42
)

# 训练模型
clf.fit(X_train, y_train)

# 预测
y_pred = clf.predict(X_test)

# 评估
print("准确率 (Accuracy):", accuracy_score(y_test, y_pred))
print("\n分类报告 (Classification Report):\n", classification_report(y_test, y_pred))
print("混淆矩阵 (Confusion Matrix):\n", confusion_matrix(y_test, y_pred))
# 5折交叉验证
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
f1_scores = cross_val_score(clf, X, y, cv=cv, scoring='f1')

# 输出交叉验证结果
print(" K-Fold 交叉验证 F1-score：")
for i, score in enumerate(f1_scores):
    print(f"Fold {i+1}: {score:.3f}")
print(f"平均 F1-score: {f1_scores.mean():.3f} ± {f1_scores.std():.3f}")


In [ ]:
import sys
sys.path.append(r"C:\Users\Adolph\anaconda3\Lib\site-packages")

import shap
explainer = shap.TreeExplainer(clf)
shap_values = explainer.shap_values(X_test)
shap.summary_plot(shap_values, X_test)


In [ ]:
import shap
import numpy as np

explainer = shap.TreeExplainer(clf)
shap_values = explainer.shap_values(X_test)  # shape: [samples, features, classes]

# 提取类别 1 的 SHAP 值
shap_values_class1 = shap_values[:, :, 1]  # shape: (1376, 7)

# 绘图（转换 features 为 numpy 并保持列名）
shap.summary_plot(
    shap_values_class1,
    features=X_test.values,
    feature_names=X_test.columns,
    plot_type="bar"
)


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# 数据
models = ['Spatial only', 'Score + Spatial']
accuracy = [0.71, 0.99]
f1_class1 = [0.24, 0.96]

x = np.arange(len(models))
width = 0.35

fig, ax = plt.subplots(figsize=(8, 5))
acc_bar = ax.bar(x - width/2, accuracy, width, label='Accuracy', color='#4CAF50')
f1_bar = ax.bar(x + width/2, f1_class1, width, label='F1-score (valuable)', color='#2196F3')

# 标注
ax.set_ylabel('Score')
ax.set_title('Performance Comparison of Structured Models')
ax.set_xticks(x)
ax.set_xticklabels(models)
ax.set_ylim(0, 1.1)
ax.legend()

for i, val in enumerate(accuracy):
    ax.text(x[i] - width/2, val + 0.02, f"{val:.2f}", ha='center')
for i, val in enumerate(f1_class1):
    ax.text(x[i] + width/2, val + 0.02, f"{val:.2f}", ha='center')

plt.tight_layout()
plt.savefig("structured_model_performance.png", dpi=300)
plt.show()


In [ ]:
def opponent_density_along_path(row):
    x0, y0 = row['passedPlayerPosX'], row['passedPlayerPosY']
    x1, y1 = row['receivedPlayerPosX'], row['receivedPlayerPosY']
    opps = [(row[f'opponentPosX{i}'], row[f'opponentPosY{i}']) for i in range(1,12)]
    
    count = 0
    for ox, oy in opps:
        if ox < 0 or oy < 0:
            continue

        if min(x0, x1) - 2 <= ox <= max(x0, x1) + 2 and min(y0, y1) - 2 <= oy <= max(y0, y1) + 2:
            count += 1
    return count

df['opp_in_path'] = df.apply(opponent_density_along_path, axis=1)


In [ ]:
import numpy as np

def point_to_segment_distance(px, py, x1, y1, x2, y2):
    # 计算点(px, py) 到线段(x1,y1)-(x2,y2) 的最短距离
    line_mag = (x2 - x1)**2 + (y2 - y1)**2
    if line_mag == 0:
        return np.sqrt((px - x1)**2 + (py - y1)**2)

    u = ((px - x1) * (x2 - x1) + (py - y1) * (y2 - y1)) / line_mag
    u = max(min(u, 1), 0)
    ix = x1 + u * (x2 - x1)
    iy = y1 + u * (y2 - y1)
    return np.sqrt((px - ix)**2 + (py - iy)**2)

def avg_opp_dist_to_pass_path(row):
    x0, y0 = row['passedPlayerPosX'], row['passedPlayerPosY']
    x1, y1 = row['receivedPlayerPosX'], row['receivedPlayerPosY']
    if x0 < 0 or y0 < 0 or x1 < 0 or y1 < 0:
        return np.nan

    opp_dists = []
    for i in range(1, 12):
        ox, oy = row.get(f'opponentPosX{i}', -1), row.get(f'opponentPosY{i}', -1)
        if ox < 0 or oy < 0:
            continue
        dist = point_to_segment_distance(ox, oy, x0, y0, x1, y1)
        opp_dists.append(dist)

    return np.mean(opp_dists) if opp_dists else np.nan
df['avg_opp_dist_to_path'] = df.apply(avg_opp_dist_to_pass_path, axis=1)


In [ ]:
import numpy as np

def calc_pass_opponent_angle(row):
    x0, y0 = row['passedPlayerPosX'], row['passedPlayerPosY']
    x1, y1 = row['receivedPlayerPosX'], row['receivedPlayerPosY']
    
    if x0 < 0 or y0 < 0 or x1 < 0 or y1 < 0:
        return np.nan

    # Compute normalized pass direction vector
    dx_pass, dy_pass = x1 - x0, y1 - y0
    norm_pass = np.sqrt(dx_pass**2 + dy_pass**2)
    if norm_pass == 0:
        return np.nan
    vec_pass = np.array([dx_pass, dy_pass]) / norm_pass

    # Find nearest opponent and compute direction vector
    min_dist = float('inf')
    vec_opp = None
    for i in range(1, 12):
        ox, oy = row.get(f'opponentPosX{i}', -1), row.get(f'opponentPosY{i}', -1)
        if ox < 0 or oy < 0:
            continue
        dx_opp, dy_opp = ox - x0, oy - y0
        dist = np.sqrt(dx_opp**2 + dy_opp**2)
        if 0 < dist < min_dist:
            min_dist = dist
            vec_opp = np.array([dx_opp, dy_opp]) / dist

    if vec_opp is None:
        return np.nan

    # Compute angle (in degrees) between pass and nearest opponent direction
    dot_product = np.dot(vec_pass, vec_opp)
    angle = np.arccos(np.clip(dot_product, -1.0, 1.0))
    return np.degrees(angle)

# Apply to each row
df['angle_pass_vs_nearest_opp'] = df.apply(calc_pass_opponent_angle, axis=1)


In [ ]:
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import pandas as pd

# 你的特征列表
features = [

    'opp_in_path','avg_opp_dist_to_path','angle_pass_vs_nearest_opp'
]

# 准备数据
df_model = df[features + ['is_valuable']].dropna()
X = df_model[features]
y = df_model['is_valuable'].astype(int)

# 划分训练/测试集
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

weight_ratio = (y == 0).sum() / (y == 1).sum()

# 定义 XGBoost 分类器
clf = XGBClassifier(
    n_estimators=300,
    max_depth=8,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    use_label_encoder=False,
    eval_metric='logloss',
    scale_pos_weight=weight_ratio,
    random_state=42
)

# 训练模型
clf.fit(X_train, y_train)

# 预测
y_pred = clf.predict(X_test)

# 评估
print("(Accuracy):", accuracy_score(y_test, y_pred))
print("\n(Classification Report):\n", classification_report(y_test, y_pred))
print(" (Confusion Matrix):\n", confusion_matrix(y_test, y_pred))


In [ ]:
import sys
sys.path.append(r"C:\Users\Adolph\anaconda3\Lib\site-packages")

import shap
explainer = shap.TreeExplainer(clf)
shap_values = explainer.shap_values(X_test)
shap.summary_plot(shap_values, X_test)


In [ ]:
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
import pandas as pd


# 特征列
features = [
    'pass_speed', 'receivedPlayerId_PassLength', 'pressure_level_num',
    'pressure_direction_num', 'pass_area_num', 'pass_direction_num', 'xT_change',
    'opp_in_path', 'avg_opp_dist_to_path', 'angle_pass_vs_nearest_opp'
]

# 删除缺失值
df_model = df[features + ['is_valuable']].dropna()
X = df_model[features]
y = df_model['is_valuable'].astype(int)

# 划分训练/测试集（用于普通训练和分类报告）
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 训练模型
clf = RandomForestClassifier(n_estimators=200, max_depth=6, random_state=42)
clf.fit(X_train, y_train)

# 预测并评估
y_pred = clf.predict(X_test)
print("单次划分结果：")
print(classification_report(y_test, y_pred))

# 5折交叉验证
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
f1_scores = cross_val_score(clf, X, y, cv=cv, scoring='f1')

# 输出交叉验证结果
print(" K-Fold 交叉验证 F1-score：")
for i, score in enumerate(f1_scores):
    print(f"Fold {i+1}: {score:.3f}")
print(f"平均 F1-score: {f1_scores.mean():.3f} ± {f1_scores.std():.3f}")


In [ ]:
import shap
explainer = shap.TreeExplainer(clf)
shap_values = explainer.shap_values(X_test)

# 类别 1（valuable）的 SHAP 可视化
shap.summary_plot(shap_values[1], X_test)


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# 平均 SHAP 值
mean_shap = np.abs(shap_values_class1).mean(axis=0)
feature_names = X_test.columns

# 按重要性排序
sorted_idx = np.argsort(mean_shap)[::-1]
sorted_names = feature_names[sorted_idx]
sorted_shap = mean_shap[sorted_idx]

# 绘图
plt.figure(figsize=(8, 5))
plt.barh(sorted_names, sorted_shap, color='#e63946')  # 红色风格
plt.gca().invert_yaxis()
plt.xlabel("Mean |SHAP value|")
plt.title("Feature Importance for Predicting Valuable Passes")
plt.tight_layout()
plt.savefig("shap_custom_bar.png", dpi=300)
plt.show()


In [ ]:
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
import pandas as pd

# 读取数据
df = pd.read_csv("label_annotation_sample (1).csv")

# 特征列
features = [
    'pass_speed', 'receivedPlayerId_PassLength', 'pressure_level_num',
    'pressure_direction_num', 'pass_area_num', 'pass_direction_num', 'xT_change',
    'opp_in_path', 'avg_opp_dist_to_path', 'angle_pass_vs_nearest_opp'
]

# 删除缺失值
df = df.dropna(subset=features + ['human_label'])

# 目标变量
X = df[features]
y = df['human_label'].astype(int)

# 划分训练/测试集（用于普通训练和分类报告）
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 训练模型
clf = RandomForestClassifier(n_estimators=200, max_depth=6, random_state=42)
clf.fit(X_train, y_train)

# 预测并评估
y_pred = clf.predict(X_test)
print("单次划分结果：")
print(classification_report(y_test, y_pred))

# 5折交叉验证
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
f1_scores = cross_val_score(clf, X, y, cv=cv, scoring='f1')

# 输出交叉验证结果
print(" K-Fold 交叉验证 F1-score：")
for i, score in enumerate(f1_scores):
    print(f"Fold {i+1}: {score:.3f}")
print(f"平均 F1-score: {f1_scores.mean():.3f} ± {f1_scores.std():.3f}")


In [ ]:
import matplotlib.pyplot as plt

# F1 scores from 5-fold CV
folds = ['Fold 1', 'Fold 2', 'Fold 3', 'Fold 4', 'Fold 5']
f1_scores = [0.923, 0.875, 0.923, 0.923, 0.769]

plt.figure(figsize=(8, 5))
bars = plt.bar(folds, f1_scores, color='#2196F3')
plt.axhline(y=sum(f1_scores)/len(f1_scores), color='gray', linestyle='--', label='Mean F1')
plt.ylim(0, 1.1)
plt.ylabel('F1-score')
plt.title('5-Fold Cross-Validation F1-score')
plt.legend()

# 标注数值
for bar, score in zip(bars, f1_scores):
    plt.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.02, f"{score:.3f}", ha='center')

plt.tight_layout()
plt.savefig("kfold_f1_scores.png", dpi=300)
plt.show()


In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

# 数据
metrics = ['Precision (0)', 'Recall (0)', 'F1 (0)', 'Precision (1)', 'Recall (1)', 'F1 (1)']
values = [0.82, 1.00, 0.90, 1.00, 0.71, 0.83]
colors = ['#4CAF50'] * 3 + ['#f44336'] * 3
legend_elements = [
    Patch(facecolor='#4CAF50', edgecolor='black', label='Not Valuable'),
    Patch(facecolor='#f44336', edgecolor='black', label='Valuable')
]

# 创建图形
plt.figure(figsize=(12, 8))
bars = plt.bar(metrics, values, color=colors, edgecolor='black', alpha=0.9)

# 添加数值标签
for bar, val in zip(bars, values):
    plt.text(bar.get_x() + bar.get_width()/2, val + 0.02, f"{val:.2f}",
             ha='center', va='bottom', fontsize=16, fontweight='bold')

# 图例
plt.legend(handles=legend_elements, loc='upper right', fontsize=16)

# 设置坐标轴和标题（直接加粗）
plt.ylim(0, 1.1)
plt.xticks(rotation=15, fontsize=16, fontweight='bold')
plt.yticks(fontsize=16, fontweight='bold')
plt.title('RandomForest on Human-Labeled Valuable Passes', fontsize=20, fontweight='bold')
plt.ylabel('Score', fontsize=18, fontweight='bold')

# 输出
plt.tight_layout()
plt.savefig("randomforest_humanlabel_metrics_allbold_fixed.png", dpi=300)
plt.show()


In [ ]:
# 初始化 SHAP 解释器
explainer = shap.TreeExplainer(clf)
shap_values = explainer.shap_values(X_test)

# 处理 shap_values：根据维度确定使用方式
if isinstance(shap_values, list):
    shap_to_plot = shap_values[1]
elif shap_values.ndim == 3:
    shap_to_plot = shap_values[:, :, 1]
else:
    shap_to_plot = shap_values

# summary_plot 分布图（推荐用 jupyter notebook 运行）
shap.summary_plot(shap_to_plot, X_test)
shap.summary_plot(shap_to_plot, X_test, plot_type="bar")

# dependence plot：特征与模型影响关系
shap.dependence_plot("xT_change", shap_to_plot, X_test)
shap.dependence_plot("pass_speed", shap_to_plot, X_test)

# force_plot 单个样本解释（适合 HTML 或 notebook 显示）
i = 3
shap.initjs()
shap.plots.force(
    explainer.expected_value[1],
    shap_values[:, :, 1][i],
    X_test.iloc[i]
)

# 分布图（点图）
shap.summary_plot(shap_to_plot, X_test, show=False)
plt.tight_layout()
plt.savefig("shap_summary_dot.png", dpi=300, bbox_inches='tight')
plt.close()



In [ ]:
def extract_frame_players_and_roles(coordinates_df, row):
    timestamp = row['timestamp']
    passer_id = row['passedPlayerId']
    team_id = row['passedPlayerTeamId']

    frame = coordinates_df[coordinates_df['timestamp'] == timestamp]
    if frame.empty or pd.isna(timestamp):
        raise ValueError(f"No coordinate data found for timestamp {timestamp}")
    frame = frame.iloc[0]

    # Extract all players with valid IDs and positions
    player_rows = []
    for i in range(40):
        pid = frame.get(f'playerId{i}', -100)
        x = frame.get(f'posX{i}', -100)
        y = frame.get(f'posY{i}', -100)
        if pid == -100 or x == -100 or y == -100:
            continue
        player_rows.append({'player_index': i, 'player_id': pid, 'posX': x, 'posY': y})

    df_players = pd.DataFrame(player_rows)

    # Retrieve teammate and opponent IDs from pass record
    teammate_ids = [row.get(f'teamMatePlayerId{i}') for i in range(1, 11)]
    opponent_ids = [row.get(f'opponentPlayerId{i}') for i in range(1, 12)]

    # Assign roles
    df_players['is_passer'] = df_players['player_id'] == passer_id
    df_players['is_teammate'] = df_players['player_id'].isin(teammate_ids)
    df_players['is_opponent'] = df_players['player_id'].isin(opponent_ids)

    return row, df_players


In [ ]:
def draw_frame_players(df_players, pass_row):
    # Set up pitch display
    plt.figure(figsize=(10, 6.5))
    plt.xlim(0, 105)
    plt.ylim(0, 68)
    plt.gca().set_facecolor('lightgreen')
    plt.title("Spatial Situation at the Time of the Pass", fontsize=14)

    receiver_id = pass_row.get('receivedPlayerId', None)

    # Draw passer
    passer = df_players[df_players['is_passer']]
    plt.scatter(passer['posX'], passer['posY'], color='blue', s=100, label='Passer')

    # Draw teammates (excluding receiver)
    teammates = df_players[df_players['is_teammate'] & (df_players['player_id'] != receiver_id)]
    plt.scatter(teammates['posX'], teammates['posY'], color='white', s=60, label='Teammates')

    # Draw receiver
    receiver = df_players[df_players['player_id'] == receiver_id]
    if not receiver.empty:
        plt.scatter(receiver['posX'], receiver['posY'], color='yellow', edgecolors='black', s=100, label='Receiver', zorder=5)

    # Draw opponents
    opponents = df_players[df_players['is_opponent']]
    plt.scatter(opponents['posX'], opponents['posY'], color='black', s=60, label='Opponents')



    # Draw pass arrow from passer to receiver
    try:
        passer_row = df_players[df_players['is_passer']].iloc[0]
        receiver_row = df_players[df_players['player_id'] == receiver_id].iloc[0]

        plt.arrow(
            passer_row['posX'], passer_row['posY'],
            receiver_row['posX'] - passer_row['posX'],
            receiver_row['posY'] - passer_row['posY'],
            width=0.3, head_width=2.2, length_includes_head=True, color='red', alpha=0.9
        )
    except:
        pass

    plt.xlabel("Field X")
    plt.ylabel("Field Y")
    plt.legend()
    plt.grid(True)
    plt.show()


In [ ]:
row = df_passes.iloc[20]  # 或 df_passes_cleaned.iloc[500]
pass_row, frame_players = extract_frame_players_and_roles(df_coordinates, row)
draw_frame_players(frame_players, pass_row)


In [ ]:
from collections import Counter
error_counter = Counter()



# 图像保存路径和数量
save_dir = r"C:\Users\Adolph\graduation thesis\image"
os.makedirs(save_dir, exist_ok=True)
num_samples = 5000  # 总生成图像数量

# 确保有 window_score，并加 is_valuable
df = df.copy()
df['is_valuable'] = (df['window_score'] > 0.5).astype(int)

# 分离样本
df_positive = df[df['is_valuable'] == 1]
df_negative = df[df['is_valuable'] == 0].sample(n=max(0, num_samples - len(df_positive)), random_state=42)
df_selected = pd.concat([df_positive, df_negative]).sample(frac=1.0, random_state=42).reset_index(drop=True)

# 绘图函数（保持原样）
def draw_pass_frame_as_image(df_players, pass_row, save_path):
    plt.figure(figsize=(10, 6.5))
    plt.xlim(0, 105)
    plt.ylim(0, 68)
    plt.gca().set_facecolor('lightgreen')
    plt.title("Spatial Situation at the Time of the Pass", fontsize=14)

    receiver_id = pass_row.get('receivedPlayerId', None)
    passer = df_players[df_players['is_passer']]
    plt.scatter(passer['posX'], passer['posY'], color='blue', s=100, label='Passer')

    teammates = df_players[df_players['is_teammate'] & (df_players['player_id'] != receiver_id)]
    plt.scatter(teammates['posX'], teammates['posY'], color='white', s=60, label='Teammates')

    receiver = df_players[df_players['player_id'] == receiver_id]
    if not receiver.empty:
        plt.scatter(receiver['posX'], receiver['posY'], color='yellow', edgecolors='black', s=100, label='Receiver', zorder=5)

    opponents = df_players[df_players['is_opponent']]
    plt.scatter(opponents['posX'], opponents['posY'], color='black', s=60, label='Opponents')

    for _, row in df_players.iterrows():
        if pd.notna(row['player_id']):
            plt.text(row['posX'], row['posY'] + 1, str(int(row['player_id'])), fontsize=8, ha='center')

    try:
        passer_row = passer.iloc[0]
        receiver_row = receiver.iloc[0]
        plt.arrow(
            passer_row['posX'], passer_row['posY'],
            receiver_row['posX'] - passer_row['posX'],
            receiver_row['posY'] - passer_row['posY'],
            width=0.3, head_width=2.2, length_includes_head=True, color='red', alpha=0.9
        )
    except:
        pass

    plt.xlabel("Field X")
    plt.ylabel("Field Y")
    plt.legend()
    plt.grid(True)
    plt.savefig(save_path, bbox_inches='tight', pad_inches=0.1)
    plt.close()

# 生成图像与标签
label_rows = []
count = 0

for idx, pass_row in df_selected.iterrows():
    ts = pass_row['timestamp']
    if pd.isna(ts):
        error_counter["timestamp NaN"] += 1
        continue
    try:
        pass_row_out, df_players = extract_frame_players_and_roles(df_coordinates, pass_row)

        img_name = f"pass_{idx:04d}.png"
        img_path = os.path.join(save_dir, img_name)
        draw_pass_frame_as_image(df_players, pass_row_out, img_path)

        label_rows.append({
            'filename': img_name,
            'is_valuable': int(pass_row['is_valuable'])
        })
        count += 1

    except Exception as e:
        error_counter[str(e)] += 1
        continue

# 保存标签文件
df_labels = pd.DataFrame(label_rows)
csv_path = os.path.join(save_dir, "pass_labels_valuable.csv")
df_labels.to_csv(csv_path, index=False)

print(f"Images generated: {count}")
print(f"Label CSV saved to: {csv_path}")
print(df_labels['is_valuable'].value_counts())
print(df_labels.head())
print("Error Summary:")
for err, count in error_counter.most_common():
    print(f"{err}: {count}")

In [ ]:
import os
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras import layers, models, optimizers, losses, metrics
from tensorflow.keras.utils import Sequence
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from sklearn.model_selection import train_test_split
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications.resnet50 import preprocess_input
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

# 图像和标签路径
image_dir = r"C:\Users\Adolph\graduation thesis\image"
label_path = os.path.join(image_dir, "pass_labels_valuable.csv")
df = pd.read_csv(label_path)

# 清理数据
df = df.dropna(subset=['is_valuable'])
df['is_valuable'] = df['is_valuable'].astype(int)

# 训练集 / 验证集划分
train_df, val_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['is_valuable'])

# 自定义图像加载器
class ImageDataset(Sequence):
    def __init__(self, df, batch_size=32, image_size=(224, 224), shuffle=True):
        self.df = df.reset_index(drop=True)
        self.batch_size = batch_size
        self.image_size = image_size
        self.shuffle = shuffle
        self.indices = np.arange(len(self.df))
        self.on_epoch_end()

    def __len__(self):
        return int(np.ceil(len(self.df) / self.batch_size))

    def __getitem__(self, index):
        batch_indices = self.indices[index*self.batch_size:(index+1)*self.batch_size]
        batch_df = self.df.iloc[batch_indices]
        images, labels = [], []
        for _, row in batch_df.iterrows():
            img_path = os.path.join(image_dir, row['filename'])
            img = load_img(img_path, target_size=self.image_size)
            img_array = preprocess_input(img_to_array(img))  # 使用 ResNet50 预处理
            images.append(img_array)
            labels.append(row['is_valuable'])
        return np.array(images), np.array(labels, dtype=np.float32)

    def on_epoch_end(self):
        if self.shuffle:
            np.random.shuffle(self.indices)

# 数据生成器
train_gen = ImageDataset(train_df)
val_gen = ImageDataset(val_df, shuffle=False)

# 构建 ResNet50 模型
def build_resnet_model(input_shape=(224, 224, 3)):
    base_model = ResNet50(weights='imagenet', include_top=False, input_shape=input_shape)
    base_model.trainable = False  # 冻结预训练层
    x = base_model.output
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dropout(0.4)(x)
    output = layers.Dense(1, activation='sigmoid')(x)
    return models.Model(inputs=base_model.input, outputs=output)

model = build_resnet_model()
model.compile(
    optimizer=optimizers.Adam(learning_rate=1e-4),
    loss=losses.BinaryCrossentropy(),
    metrics=[
        metrics.BinaryAccuracy(),
        metrics.Precision(),
        metrics.Recall(),
        metrics.AUC()
    ]
)

# 训练
callbacks = [
    EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True),
    ModelCheckpoint("best_resnet_model.h5", save_best_only=True)
]

model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=20,
    class_weight={0: 1.0, 1: 3.0},
    callbacks=callbacks,
    verbose=1
)

# 保存最终模型
model.save("resnet50_cnn_is_valuable_final.h5")


In [ ]:
def extract_frame_players_and_roles(coordinates_df, row):
    timestamp = row['timestamp']
    passer_id = row['passedPlayerId']
    team_id = row['passedPlayerTeamId']

    frame = coordinates_df[coordinates_df['timestamp'] == timestamp]
    if frame.empty or pd.isna(timestamp):
        raise ValueError(f"No coordinate data found for timestamp {timestamp}")
    frame = frame.iloc[0]

    # Extract all players with valid IDs and positions
    player_rows = []
    for i in range(40):
        pid = frame.get(f'playerId{i}', -100)
        x = frame.get(f'posX{i}', -100)
        y = frame.get(f'posY{i}', -100)
        if pid == -100 or x == -100 or y == -100:
            continue
        player_rows.append({'player_index': i, 'player_id': pid, 'posX': x, 'posY': y})

    df_players = pd.DataFrame(player_rows)

    # Retrieve teammate and opponent IDs from pass record
    teammate_ids = [row.get(f'teamMatePlayerId{i}') for i in range(1, 11)]
    opponent_ids = [row.get(f'opponentPlayerId{i}') for i in range(1, 12)]

    # Assign roles
    df_players['is_passer'] = df_players['player_id'] == passer_id
    df_players['is_teammate'] = df_players['player_id'].isin(teammate_ids)
    df_players['is_opponent'] = df_players['player_id'].isin(opponent_ids)

    return row, df_players


In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import Circle, FancyArrow
import numpy as np

def draw_frame_players_beautiful(df_players, pass_row):
    fig, ax = plt.subplots(figsize=(10, 6.5))
    ax.set_xlim(0, 105)
    ax.set_ylim(0, 68)
    ax.set_facecolor('#99ff99')  # 柔和背景色（淡绿色）
    ax.set_title("Tactical Context of the Pass", fontsize=16, fontweight='bold')

    # 获取传球者和接球者位置
    passer = df_players[df_players['is_passer']]
    x0, y0 = passer['posX'].values[0], passer['posY'].values[0]
    if pass_row.get('isSucceeded') == 0:
        x1 = pass_row.get('receivedPlayerPosXh', -1)
        y1 = pass_row.get('receivedPlayerPosYh', -1)
    else:
        x1 = pass_row.get('receivedPlayerPosX', -1)
        y1 = pass_row.get('receivedPlayerPosY', -1)

    # 球员绘制
    ax.scatter(x0, y0, color='#1e88e5', s=120, label='Passer', zorder=5)
    ax.scatter(df_players[df_players['is_teammate']]['posX'],
               df_players[df_players['is_teammate']]['posY'],
               color='white', edgecolors='gray', s=80, label='Teammates', zorder=3)
    ax.scatter(df_players[df_players['is_opponent']]['posX'],
               df_players[df_players['is_opponent']]['posY'],
               color='black', s=80, label='Opponents', zorder=3)
    if x1 > 0 and y1 > 0:
        ax.scatter(x1, y1, color='gold', edgecolors='black', s=120, label='Receiver', zorder=6)

        # 传球箭头
        ax.add_patch(FancyArrow(x0, y0, x1 - x0, y1 - y0,
                                width=0.3, head_width=2.2, head_length=2,
                                length_includes_head=True, color='crimson', alpha=0.85, zorder=4))

        # ---------- opp_in_path 可视化 ----------
        dx, dy = x1 - x0, y1 - y0
        norm = np.sqrt(dx**2 + dy**2)
        if norm > 0:
            dx, dy = dx / norm, dy / norm
            normal = np.array([-dy, dx])
            for _, row in df_players[df_players['is_opponent']].iterrows():
                ox, oy = row['posX'], row['posY']
                vec = np.array([ox - x0, oy - y0])
                proj_len = np.dot(vec, [dx, dy])
                dist_to_path = abs(np.dot(vec, normal))
                if 0 <= proj_len <= norm and dist_to_path <= 2:
                    ax.scatter(ox, oy, s=200, facecolors='none', edgecolors='purple',
                               linewidths=2, label='In Path' if 'In Path' not in ax.get_legend_handles_labels()[1] else "", zorder=6)

        # ---------- angle_pass_vs_nearest_opp 可视化 ----------
        vec_pass = np.array([x1 - x0, y1 - y0])
        norm_pass = np.linalg.norm(vec_pass)
        if norm_pass > 0:
            vec_pass = vec_pass / norm_pass
            min_dist = float('inf')
            nearest_vec = None
            for _, row in df_players[df_players['is_opponent']].iterrows():
                dxo, dyo = row['posX'] - x0, row['posY'] - y0
                dist = np.sqrt(dxo**2 + dyo**2)
                if dist > 0 and dist < min_dist:
                    min_dist = dist
                    nearest_vec = np.array([dxo, dyo]) / dist
            if nearest_vec is not None:
                ax.add_patch(FancyArrow(x0, y0, nearest_vec[0]*15, nearest_vec[1]*15,
                                        width=0.2, head_width=1.5, color='royalblue', alpha=0.6,
                                        label='Angle to Nearest Opponent'))

    # ---------- Pressure Area 可视化 ----------
    pressure_level = pass_row.get("pressure_level_num", -1)
    pressure_colors = {0: "#bbdefb", 1: "#ffe082", 2: "#ef9a9a"}
    pressure_labels = {0: "Low Pressure", 1: "Medium Pressure", 2: "High Pressure"}
    color = pressure_colors.get(pressure_level, "gray")
    label = pressure_labels.get(pressure_level, "Pressure Area")
    ax.add_patch(Circle((x0, y0), radius=10, color=color, alpha=0.3, label=label))

    ax.set_xlabel("Field X", fontsize=12)
    ax.set_ylabel("Field Y", fontsize=12)
    ax.legend(loc='upper right', frameon=True, framealpha=0.95, edgecolor='gray', fancybox=True)
    ax.grid(True, linestyle='--', alpha=0.5)
    plt.tight_layout()
    plt.show()


In [ ]:
pass_row, frame_players = extract_frame_players_and_roles(df_coordinates, row)
draw_frame_players_beautiful(frame_players, pass_row)
row = df_passes.iloc[180]


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Circle
from collections import Counter

error_counter = Counter()

# 图像保存路径和数量
save_dir = r"C:\Users\Adolph\Desktop\graduation thesis\image"
os.makedirs(save_dir, exist_ok=True)
num_samples = 4000

# 确保有 window_score，并加 is_valuable
df = df.copy()
df['is_valuable'] = (df['window_score'] > 0.5).astype(int)

# 分离样本
df_positive = df[df['is_valuable'] == 1]
df_negative = df[df['is_valuable'] == 0].sample(n=max(0, num_samples - len(df_positive)), random_state=42)
df_selected = pd.concat([df_positive, df_negative]).sample(frac=1.0, random_state=42).reset_index(drop=True)

# 美化后的绘图函数
def draw_pass_frame_as_image(df_players, pass_row, save_path):
    plt.figure(figsize=(10, 6.5))
    ax = plt.gca()
    ax.set_xlim(0, 105)
    ax.set_ylim(0, 68)
    ax.set_facecolor('#99ff99')
    plt.title("Spatial Situation at the Time of the Pass", fontsize=14)

    # 获取坐标
    passer = df_players[df_players['is_passer']]
    x0, y0 = passer['posX'].values[0], passer['posY'].values[0]
    x1 = pass_row.get('receivedPlayerPosX', pass_row.get('receivedPlayerPosXh', -1))
    y1 = pass_row.get('receivedPlayerPosY', pass_row.get('receivedPlayerPosYh', -1))

    # 球员散点图
    ax.scatter(x0, y0, color='blue', s=100, label='Passer')
    ax.scatter(df_players[df_players['is_teammate']]['posX'],
               df_players[df_players['is_teammate']]['posY'],
               color='lightgray', s=60, label='Teammates')
    ax.scatter(df_players[df_players['is_opponent']]['posX'],
               df_players[df_players['is_opponent']]['posY'],
               color='black', s=60, label='Opponents')

    if x1 > 0 and y1 > 0:
        ax.scatter(x1, y1, color='gold', edgecolors='black', s=100, label='Receiver', zorder=5)

        # 传球箭头
        ax.arrow(x0, y0, x1 - x0, y1 - y0, width=0.3, head_width=2.2,
                 length_includes_head=True, color='red', alpha=0.9)

        # 路径封堵判断
        dx, dy = x1 - x0, y1 - y0
        norm = np.sqrt(dx**2 + dy**2)
        if norm > 0:
            dx, dy = dx / norm, dy / norm
            normal = np.array([-dy, dx])
            opps = df_players[df_players['is_opponent']]
            for _, row in opps.iterrows():
                ox, oy = row['posX'], row['posY']
                vec = np.array([ox - x0, oy - y0])
                proj = np.dot(vec, [dx, dy])
                dist = abs(np.dot(vec, normal))
                if 0 <= proj <= norm and dist <= 2:
                    ax.scatter(ox, oy, s=160, facecolors='none', edgecolors='purple',
                               linewidths=2, label='In Path' if 'In Path' not in ax.get_legend_handles_labels()[1] else "")

        # 角度可视化（蓝色箭头）
        nearest_vec = None
        min_dist = float('inf')
        for _, row in opps.iterrows():
            dxo = row['posX'] - x0
            dyo = row['posY'] - y0
            dist = np.hypot(dxo, dyo)
            if dist > 0 and dist < min_dist:
                min_dist = dist
                nearest_vec = np.array([dxo, dyo]) / dist
        if nearest_vec is not None:
            ax.arrow(x0, y0, nearest_vec[0]*12, nearest_vec[1]*12,
                     width=0.2, head_width=1.2, color='blue', alpha=0.6,
                     label='Angle to Nearest Opponent' if 'Angle to Nearest Opponent' not in ax.get_legend_handles_labels()[1] else "")

    # Pressure area 圆圈颜色随等级变化
    pressure = pass_row.get('pressure_level_num', 0)
    color = {0: 'grey', 1: '#a0c4ff', 2: 'red'}.get(pressure, 'gray')
    circle = Circle((x0, y0), radius=6, color=color, alpha=0.15, label=f'Pressure Level {pressure}')
    ax.add_patch(circle)

    plt.xlabel("Field X")
    plt.ylabel("Field Y")
    plt.legend()
    plt.grid(True)
    plt.savefig(save_path, bbox_inches='tight', pad_inches=0.1)
    plt.close()

# 图像生成主循环
label_rows = []
count = 0

for idx, pass_row in df_selected.iterrows():
    ts = pass_row['timestamp']
    if pd.isna(ts):
        error_counter["timestamp NaN"] += 1
        continue
    try:
        pass_row_out, df_players = extract_frame_players_and_roles(df_coordinates, pass_row)

        img_name = f"pass_{idx:04d}.png"
        img_path = os.path.join(save_dir, img_name)
        draw_pass_frame_as_image(df_players, pass_row_out, img_path)

        label_rows.append({
            'filename': img_name,
            'is_valuable': int(pass_row['is_valuable'])
        })
        count += 1

    except Exception as e:
        error_counter[str(e)] += 1
        continue

# 保存标签文件
df_labels = pd.DataFrame(label_rows)
csv_path = os.path.join(save_dir, "pass_labels_valuable.csv")
df_labels.to_csv(csv_path, index=False)

print(f"Images generated: {count}")
print(f"Label CSV saved to: {csv_path}")
print(df_labels['is_valuable'].value_counts())
print("Error Summary:")
for err, cnt in error_counter.most_common():
    print(f"{err}: {cnt}")


In [ ]:
import os
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras import layers, models, optimizers, losses, metrics
from tensorflow.keras.utils import Sequence
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from sklearn.model_selection import train_test_split
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications.resnet50 import preprocess_input
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

# 图像和标签路径
image_dir = r"C:\Users\Adolph\Desktop\graduation thesis\image"
label_path = os.path.join(image_dir, "pass_labels_valuable.csv")
df = pd.read_csv(label_path)

# 清理数据
df = df.dropna(subset=['is_valuable'])
df['is_valuable'] = df['is_valuable'].astype(int)

# 训练集 / 验证集划分
train_df, val_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['is_valuable'])

# 自定义图像加载器
class ImageDataset(Sequence):
    def __init__(self, df, batch_size=32, image_size=(224, 224), shuffle=True):
        self.df = df.reset_index(drop=True)
        self.batch_size = batch_size
        self.image_size = image_size
        self.shuffle = shuffle
        self.indices = np.arange(len(self.df))
        self.on_epoch_end()

    def __len__(self):
        return int(np.ceil(len(self.df) / self.batch_size))

    def __getitem__(self, index):
        batch_indices = self.indices[index*self.batch_size:(index+1)*self.batch_size]
        batch_df = self.df.iloc[batch_indices]
        images, labels = [], []
        for _, row in batch_df.iterrows():
            img_path = os.path.join(image_dir, row['filename'])
            img = load_img(img_path, target_size=self.image_size)
            img_array = preprocess_input(img_to_array(img))  # 使用 ResNet50 预处理
            images.append(img_array)
            labels.append(row['is_valuable'])
        return np.array(images), np.array(labels, dtype=np.float32)

    def on_epoch_end(self):
        if self.shuffle:
            np.random.shuffle(self.indices)

# 数据生成器
train_gen = ImageDataset(train_df)
val_gen = ImageDataset(val_df, shuffle=False)

# 构建 ResNet50 模型
def build_resnet_model(input_shape=(224, 224, 3)):
    base_model = ResNet50(weights='imagenet', include_top=False, input_shape=input_shape)
    base_model.trainable = False  # 冻结预训练层
    x = base_model.output
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dropout(0.4)(x)
    output = layers.Dense(1, activation='sigmoid')(x)
    return models.Model(inputs=base_model.input, outputs=output)

model = build_resnet_model()
model.compile(
    optimizer=optimizers.Adam(learning_rate=1e-4),
    loss=losses.BinaryCrossentropy(),
    metrics=[
        metrics.BinaryAccuracy(),
        metrics.Precision(),
        metrics.Recall(),
        metrics.AUC()
    ]
)

# 训练
callbacks = [
    EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True),
    ModelCheckpoint("best_resnet_model.h5", save_best_only=True)
]

model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=20,
    class_weight={0: 1.0, 1: 3.0},
    callbacks=callbacks,
    verbose=1
)

# 保存最终模型
model.save("resnet50_cnn_is_valuable_final.h5")


In [ ]:
import os
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras import layers, models, optimizers, metrics
from tensorflow.keras.utils import Sequence
from tensorflow.keras.preprocessing.image import load_img, img_to_array, ImageDataGenerator
from sklearn.model_selection import train_test_split
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications.resnet50 import preprocess_input
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

# 图像路径设置
image_dir = r"C:\Users\Adolph\Desktop\graduation thesis\image"
label_path = os.path.join(image_dir, "pass_labels_valuable.csv")
df = pd.read_csv(label_path)

# 清洗数据
df = df.dropna(subset=['is_valuable'])
df['is_valuable'] = df['is_valuable'].astype(int)

# 划分数据集
train_df, val_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['is_valuable'])

# 图像增强器
datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=10,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True
)

# 自定义数据加载器
class ImageDataset(Sequence):
    def __init__(self, df, batch_size=32, image_size=(224, 224), shuffle=True, augment=False):
        self.df = df.reset_index(drop=True)
        self.batch_size = batch_size
        self.image_size = image_size
        self.shuffle = shuffle
        self.augment = augment
        self.indices = np.arange(len(self.df))
        self.on_epoch_end()

    def __len__(self):
        return int(np.ceil(len(self.df) / self.batch_size))

    def __getitem__(self, index):
        batch_indices = self.indices[index * self.batch_size:(index + 1) * self.batch_size]
        batch_df = self.df.iloc[batch_indices]
        images, labels = [], []
        for _, row in batch_df.iterrows():
            img_path = os.path.join(image_dir, row['filename'])
            img = load_img(img_path, target_size=self.image_size)
            img_array = img_to_array(img)
            if self.augment:
                img_array = datagen.random_transform(img_array)
            img_array = preprocess_input(img_array)
            images.append(img_array)
            labels.append(row['is_valuable'])
        return np.array(images), np.array(labels, dtype=np.float32)

    def on_epoch_end(self):
        if self.shuffle:
            np.random.shuffle(self.indices)

# 加载数据生成器
train_gen = ImageDataset(train_df, augment=True)
val_gen = ImageDataset(val_df, shuffle=False)

# 构建模型（解冻最后30层）
def build_resnet_model(input_shape=(224, 224, 3)):
    base_model = ResNet50(weights='imagenet', include_top=False, input_shape=input_shape)
    base_model.trainable = True

    for layer in base_model.layers[:-30]:
        layer.trainable = False
    for layer in base_model.layers[-30:]:
        if isinstance(layer, layers.BatchNormalization):
            layer.trainable = False

    x = base_model.output
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dropout(0.4)(x)
    output = layers.Dense(1, activation='sigmoid')(x)
    return models.Model(inputs=base_model.input, outputs=output)

# Focal Loss 实现
def focal_loss(gamma=2., alpha=0.25):
    def loss(y_true, y_pred):
        epsilon = tf.keras.backend.epsilon()
        y_pred = tf.clip_by_value(y_pred, epsilon, 1. - epsilon)
        pt = tf.where(tf.equal(y_true, 1), y_pred, 1 - y_pred)
        return -tf.reduce_mean(alpha * tf.pow(1. - pt, gamma) * tf.math.log(pt))
    return loss

# 构建和编译模型
model = build_resnet_model()
model.compile(
    optimizer=optimizers.Adam(learning_rate=1e-5),  # 微调时小学习率
    loss=focal_loss(gamma=2.0, alpha=0.25),
    metrics=[
        metrics.BinaryAccuracy(name='binary_accuracy'),
        metrics.Precision(name='precision'),
        metrics.Recall(name='recall'),
        metrics.AUC(name='auc')
    ]
)

# 回调
callbacks = [
    EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True),
    ModelCheckpoint("best_resnet_model_focal_finetune.h5", save_best_only=True)
]

# 训练
model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=20,
    class_weight={0: 1.0, 1: 3.0},  # 可以试试 balanced 权重或更微调
    callbacks=callbacks,
    verbose=1
)

# 保存最终模型
model.save("resnet50_focal_finetuned_final.h5")


In [ ]:
import os
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
import cv2
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from tensorflow.keras.applications.resnet50 import preprocess_input

# ========== 配置 ==========
image_dir = r"C:\Users\Adolph\Desktop\graduation thesis\cnn_pass_images"
model_path = "resnet50_focal_finetuned_final.h5"
image_filename = "cnn_ready_pass_1.png"  # 替换成你要看的图
last_conv_layer_name = "conv5_block3_out"  # ResNet50 最后一层卷积层

# ========== 加载模型 ==========
model = tf.keras.models.load_model(model_path, compile=False)

# ========== 图像加载与预处理 ==========
def load_and_preprocess_image(img_path, target_size=(224, 224)):
    img = load_img(img_path, target_size=target_size)
    raw_img = img.copy()
    img_array = img_to_array(img)
    img_array = preprocess_input(img_array)
    return np.expand_dims(img_array, axis=0), raw_img

img_path = os.path.join(image_dir, image_filename)
img_tensor, raw_img = load_and_preprocess_image(img_path)

# ========== Grad-CAM 核心函数 ==========
def make_gradcam_heatmap(img_array, model, last_conv_layer_name):
    grad_model = tf.keras.models.Model(
        [model.inputs],
        [model.get_layer(last_conv_layer_name).output, model.output]
    )

    with tf.GradientTape() as tape:
        conv_outputs, predictions = grad_model(img_array)
        loss = predictions[:, 0]

    grads = tape.gradient(loss, conv_outputs)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))

    conv_outputs = conv_outputs[0]
    heatmap = tf.reduce_sum(tf.multiply(pooled_grads, conv_outputs), axis=-1)

    heatmap = tf.maximum(heatmap, 0) / tf.reduce_max(heatmap + tf.keras.backend.epsilon())
    return heatmap.numpy()

# ========== 高质量叠加显示 ==========
def display_gradcam(heatmap, original_img, alpha=0.5, cmap='jet'):
    heatmap = np.uint8(255 * heatmap)
    heatmap_resized = cv2.resize(heatmap, (original_img.size[0], original_img.size[1]))
    heatmap_colored = cv2.applyColorMap(heatmap_resized, cv2.COLORMAP_JET)

    img = np.array(original_img)
    if img.shape[-1] == 4:
        img = img[..., :3]

    # 可选：增强亮度
    heatmap_colored = cv2.convertScaleAbs(heatmap_colored, alpha=1.2, beta=30)

    superimposed_img = cv2.addWeighted(img, 1 - alpha, heatmap_colored, alpha, 0)
    
    plt.figure(figsize=(6, 6))
    plt.imshow(superimposed_img)
    plt.axis('off')
    plt.title("Grad-CAM: Model Attention on Valuable Pass")
    plt.show()


# ========== 运行流程 ==========
heatmap = make_gradcam_heatmap(img_tensor, model, last_conv_layer_name)
display_gradcam(heatmap, raw_img)


In [ ]:
pip install opencv-python-headless


In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import FancyArrow

# ========== 1. 提取球员身份与坐标 ==========
def extract_frame_players_and_roles(coordinates_df, row):
    timestamp = row['timestamp']
    passer_id = row['passedPlayerId']
    team_id = row['passedPlayerTeamId']

    frame = coordinates_df[coordinates_df['timestamp'] == timestamp]
    if frame.empty or pd.isna(timestamp):
        raise ValueError(f"No coordinate data found for timestamp {timestamp}")
    frame = frame.iloc[0]

    player_rows = []
    for i in range(40):
        pid = frame.get(f'playerId{i}', -100)
        x = frame.get(f'posX{i}', -100)
        y = frame.get(f'posY{i}', -100)
        if pid == -100 or x == -100 or y == -100:
            continue
        player_rows.append({'player_index': i, 'player_id': pid, 'posX': x, 'posY': y})

    df_players = pd.DataFrame(player_rows)

    teammate_ids = [row.get(f'teamMatePlayerId{i}') for i in range(1, 11)]
    opponent_ids = [row.get(f'opponentPlayerId{i}') for i in range(1, 12)]

    df_players['is_passer'] = df_players['player_id'] == passer_id
    df_players['is_teammate'] = df_players['player_id'].isin(teammate_ids)
    df_players['is_opponent'] = df_players['player_id'].isin(opponent_ids)

    return row, df_players

# ========== 2. 绘图函数（大图 1000×650 用于 Grad-CAM） ==========
def draw_cnn_ready_pass_scene(df_players, pass_row, save_path=None):
    fig, ax = plt.subplots(figsize=(10, 6.5))  # ✅ 大图比例
    ax.set_xlim(0, 105)
    ax.set_ylim(0, 68)
    ax.set_facecolor('#60B860')  # 球场绿色
    ax.axis('off')

    # 传球者
    passer = df_players[df_players['is_passer']]
    x0, y0 = passer['posX'].values[0], passer['posY'].values[0]
    ax.scatter(x0, y0, color='blue', s=80, zorder=3)

    # 接球者
    if pass_row.get('isSucceeded') == 0:
        x1 = pass_row.get('receivedPlayerPosXh', -1)
        y1 = pass_row.get('receivedPlayerPosYh', -1)
    else:
        x1 = pass_row.get('receivedPlayerPosX', -1)
        y1 = pass_row.get('receivedPlayerPosY', -1)

    if x1 > 0 and y1 > 0:
        ax.scatter(x1, y1, color='yellow', s=80, edgecolors='black', zorder=3)
        ax.add_patch(FancyArrow(x0, y0, x1 - x0, y1 - y0,
                                width=0.3, head_width=2.0, head_length=2.5,
                                length_includes_head=True, color='red', zorder=2))

    # 队友
    ax.scatter(df_players[df_players['is_teammate']]['posX'],
               df_players[df_players['is_teammate']]['posY'],
               color='white', s=60, zorder=1)

    # 对手
    ax.scatter(df_players[df_players['is_opponent']]['posX'],
               df_players[df_players['is_opponent']]['posY'],
               color='black', s=60, zorder=1)

    # 保存或展示
    if save_path:
        plt.savefig(save_path, dpi=100, bbox_inches='tight', pad_inches=0, facecolor=ax.get_facecolor())  # ✅ 改 dpi 为 100
        plt.close()
    else:
        plt.tight_layout()
        plt.show()

# ========== 3. 批量生成图像 ==========
image_save_dir = r"C:\Users\Adolph\Desktop\graduation thesis\cnn_pass_images"
os.makedirs(image_save_dir, exist_ok=True)

for i in range(5):
    row = df_passes.iloc[i]
    try:
        pass_row, frame_players = extract_frame_players_and_roles(df_coordinates, row)
        save_path = os.path.join(image_save_dir, f"cnn_ready_pass_{i}.png")
        draw_cnn_ready_pass_scene(frame_players, pass_row, save_path=save_path)
        print(f"✅ Saved: {save_path}")
    except Exception as e:
        print(f"⚠️ Skipped {i} due to error: {e}")


In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import cv2
import tensorflow as tf
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.image import load_img, img_to_array

# ======== 1. 加载模型 ========
model_path = r"C:\Users\Adolph\graduation thesis\resnet50_cnn_is_valuable_final.h5"
model = load_model(model_path)
last_conv_layer_name = "conv5_block3_out"  # ResNet50 最后一个卷积层名

# ======== 2. Grad-CAM 核心函数（简化版）========
def compute_gradcam(model, image_tensor, last_conv_layer_name):
    grad_model = tf.keras.models.Model(
        [model.inputs],
        [model.get_layer(last_conv_layer_name).output, model.output]
    )

    with tf.GradientTape() as tape:
        conv_outputs, predictions = grad_model(image_tensor)
        pred_index = tf.argmax(predictions[0])
        class_output = predictions[:, pred_index]

    grads = tape.gradient(class_output, conv_outputs)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
    conv_outputs = conv_outputs[0]
    heatmap = tf.reduce_sum(tf.multiply(pooled_grads, conv_outputs), axis=-1)

    heatmap = tf.maximum(heatmap, 0)
    heatmap /= tf.reduce_max(heatmap) + tf.keras.backend.epsilon()
    return heatmap.numpy()

# ======== 3. 图像叠加函数 ========
def overlay_gradcam_on_image(img_path, heatmap, alpha=0.4, colormap=cv2.COLORMAP_JET, output_path=None):
    img = cv2.imread(img_path)
    img = cv2.resize(img, (224, 224))
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    heatmap_resized = cv2.resize(heatmap, (224, 224))
    heatmap_resized = np.uint8(255 * heatmap_resized)
    heatmap_color = cv2.applyColorMap(heatmap_resized, colormap)

    superimposed_img = cv2.addWeighted(img_rgb, 1 - alpha, heatmap_color, alpha, 0)

    plt.figure(figsize=(6, 4))
    plt.imshow(superimposed_img)
    plt.title("Grad-CAM: Model Attention on Valuable Pass", fontsize=14)
    plt.axis('off')
    if output_path:
        plt.savefig(output_path, bbox_inches='tight', pad_inches=0)
        plt.close()
    else:
        plt.show()

# ======== 4. 图像处理路径与批量执行 ========
img_dir = r"C:\Users\Adolph\Desktop\graduation thesis\cnn_pass_images"
save_dir = os.path.join(img_dir, "gradcam_output")
os.makedirs(save_dir, exist_ok=True)

image_files = [f for f in os.listdir(img_dir) if f.endswith('.png') and f.startswith("cnn_ready")]
print(f"Found {len(image_files)} images to process.")

for fname in image_files:
    try:
        path = os.path.join(img_dir, fname)
        print(f"\U0001F50D Processing: {path}")

        # 加载图像
        img = load_img(path, target_size=(224, 224))
        img_array = img_to_array(img)
        img_tensor = np.expand_dims(img_array, axis=0)
        img_tensor = tf.keras.applications.resnet50.preprocess_input(img_tensor)

        # 计算热力图
        heatmap = compute_gradcam(model, img_tensor, last_conv_layer_name)

        # 可视化 & 保存
        save_path = os.path.join(save_dir, f"gradcam_{fname}")
        overlay_gradcam_on_image(path, heatmap, output_path=save_path)
        print(f"✅ Saved to {save_path}")

    except Exception as e:
        print(f"❌ Failed on {fname}: {e}")


In [ ]:
print(model.output)


In [ ]:
def extract_frame_players_and_roles(coordinates_df, passes_df, row_idx):
    # Extract pass event row
    row = passes_df.loc[row_idx]
    timestamp = row['normal_time']
    passer_id = row['passedPlayerId']
    team_id = row['passedPlayerTeamId']

    # Get corresponding coordinate frame at the same timestamp
    frame = coordinates_df[coordinates_df['timestamp'] == timestamp]
    if frame.empty:
        raise ValueError(f"No coordinate data found for timestamp {timestamp}")
    frame = frame.iloc[0]

    # Extract all players with valid IDs and positions
    player_rows = []
    for i in range(40):
        pid = frame.get(f'playerId{i}', -100)
        x = frame.get(f'posX{i}', -100)
        y = frame.get(f'posY{i}', -100)
        if pid == -100 or x == -100 or y == -100:
            continue
        player_rows.append({'player_index': i, 'player_id': pid, 'posX': x, 'posY': y})

    df_players = pd.DataFrame(player_rows)

    # Retrieve teammate and opponent IDs from pass record
    teammate_ids = [row.get(f'teamMatePlayerId{i}') for i in range(1, 11)]
    opponent_ids = [row.get(f'opponentPlayerId{i}') for i in range(1, 12)]

    # Assign roles
    df_players['is_passer'] = df_players['player_id'] == passer_id
    df_players['is_teammate'] = df_players['player_id'].isin(teammate_ids)
    df_players['is_opponent'] = df_players['player_id'].isin(opponent_ids)

    return row, df_players


In [ ]:
def draw_frame_players(df_players, pass_row):
    # Set up pitch display
    plt.figure(figsize=(10, 6.5))
    plt.xlim(0, 105)
    plt.ylim(0, 68)
    plt.gca().set_facecolor('lightgreen')
    plt.title("Spatial Situation at the Time of the Pass", fontsize=14)

    receiver_id = pass_row.get('receivedPlayerId', None)

    # Draw passer
    passer = df_players[df_players['is_passer']]
    plt.scatter(passer['posX'], passer['posY'], color='blue', s=100, label='Passer')

    # Draw teammates (excluding receiver)
    teammates = df_players[df_players['is_teammate'] & (df_players['player_id'] != receiver_id)]
    plt.scatter(teammates['posX'], teammates['posY'], color='white', s=60, label='Teammates')

    # Draw receiver
    receiver = df_players[df_players['player_id'] == receiver_id]
    if not receiver.empty:
        plt.scatter(receiver['posX'], receiver['posY'], color='yellow', edgecolors='black', s=100, label='Receiver', zorder=5)

    # Draw opponents
    opponents = df_players[df_players['is_opponent']]
    plt.scatter(opponents['posX'], opponents['posY'], color='black', s=60, label='Opponents')

    # Annotate player IDs
    for _, row in df_players.iterrows():
        if pd.notna(row['player_id']):
            plt.text(row['posX'], row['posY'] + 1, str(int(row['player_id'])), fontsize=8, ha='center')

    # Draw pass arrow from passer to receiver
    try:
        passer_row = df_players[df_players['is_passer']].iloc[0]
        receiver_row = df_players[df_players['player_id'] == receiver_id].iloc[0]

        plt.arrow(
            passer_row['posX'], passer_row['posY'],
            receiver_row['posX'] - passer_row['posX'],
            receiver_row['posY'] - passer_row['posY'],
            width=0.3, head_width=2.2, length_includes_head=True, color='red', alpha=0.9
        )
    except:
        pass

    plt.xlabel("Field X")
    plt.ylabel("Field Y")
    plt.legend()
    plt.grid(True)
    plt.show()


In [ ]:
pass_row, frame_players = extract_frame_players_and_roles(df_coordinates, df_passes, 200)
draw_frame_players(frame_players, pass_row)


In [ ]:
import matplotlib.pyplot as plt

epochs = list(range(1, 15))  # 你目前有 14 个 epoch 的数据

val_auc = [0.6633, 0.7001, 0.7123, 0.7213, 0.7157, 0.7378, 0.7354, 0.7174, 0.7594, 0.7533, 0.7753, 0.7648, 0.7600, 0.7743]
val_accuracy = [0.7075, 0.7200, 0.7312, 0.7200, 0.7237, 0.7387, 0.7275, 0.6400, 0.7513, 0.7300, 0.7613, 0.6975, 0.7400, 0.7487]
val_precision = [0.4615, 0.5577, 0.5806, 0.5139, 0.5254, 0.5528, 0.5316, 0.4208, 0.7067, 0.5330, 0.6822, 0.4831, 0.5509, 0.5639]
val_recall = [0.1043, 0.1261, 0.2348, 0.4826, 0.4043, 0.4783, 0.4391, 0.6696, 0.2304, 0.4913, 0.3174, 0.7435, 0.5174, 0.5565]

plt.figure(figsize=(10, 6))
plt.plot(epochs, val_auc, label='Val AUC', marker='o')
plt.plot(epochs, val_accuracy, label='Val Accuracy', marker='s')
plt.plot(epochs, val_precision, label='Val Precision', marker='^')
plt.plot(epochs, val_recall, label='Val Recall', marker='x')

plt.xlabel('Epoch')
plt.ylabel('Metric Value')
plt.title('CNN Validation Metrics Across Epochs')
plt.ylim(0, 1.05)
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend()
plt.tight_layout()
plt.savefig("cnn_validation_metrics.png", dpi=300)
plt.show()


In [ ]:
import matplotlib.pyplot as plt

epochs = list(range(1, 11))

val_auc = [0.5000] * 10
val_accuracy = [0.3500] * 10
val_precision = [0.3500] * 10
val_recall = [1.0000] * 10

plt.figure(figsize=(10, 6))
plt.plot(epochs, val_auc, label='Val AUC', marker='o')
plt.plot(epochs, val_accuracy, label='Val Accuracy', marker='s')
plt.plot(epochs, val_precision, label='Val Precision', marker='^')
plt.plot(epochs, val_recall, label='Val Recall', marker='x')

plt.xlabel('Epoch')
plt.ylabel('Metric Value')
plt.title('CNN Validation Metrics (Basic Images)')
plt.ylim(0, 1.05)
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend()
plt.tight_layout()
plt.savefig("cnn_validation_metrics_basic.png", dpi=300)
plt.show()


In [ ]:
import matplotlib.pyplot as plt

# 数据
epochs = list(range(1, 15))
val_auc = [0.6633, 0.7001, 0.7123, 0.7213, 0.7157, 0.7378, 0.7354, 0.7174, 0.7594, 0.7533, 0.7753, 0.7648, 0.7600, 0.7743]
val_accuracy = [0.7075, 0.7200, 0.7312, 0.7200, 0.7237, 0.7387, 0.7275, 0.6400, 0.7513, 0.7300, 0.7613, 0.6975, 0.7400, 0.7487]

# 字体设置
plt.rcParams.update({
    'font.size': 16,
    'font.weight': 'bold',
    'axes.labelweight': 'bold',
    'axes.titlesize': 20,
    'axes.titleweight': 'bold'
})

# 画图
plt.figure(figsize=(10, 6))
plt.plot(epochs, val_auc, marker='o', linewidth=2.5, color='#1f77b4')
plt.plot(epochs, val_accuracy, marker='s', linewidth=2.5, color='#ff7f0e')

# 指定中后段位置（epoch 11）
label_epoch = 11
x_pos = epochs[label_epoch]
y_auc = val_auc[label_epoch]
y_acc = val_accuracy[label_epoch]

# 添加标签：AUC 往上偏，Accuracy 往下偏
plt.text(x_pos, y_auc + 0.025, 'Area Under Curve',
         fontsize=14, fontweight='bold', color='#1f77b4')

plt.text(x_pos, y_acc - 0.04, 'Validation Accuracy',
         fontsize=14, fontweight='bold', color='#ff7f0e')

# 图像设置
plt.xlabel('Epoch', fontsize=16, fontweight='bold')
plt.ylabel('Metric Value', fontsize=16, fontweight='bold')
plt.title('CNN Validation Performance Over Epochs', fontsize=20, fontweight='bold')
plt.ylim(0, 1.05)
plt.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()

# 保存图像
plt.savefig("cnn_validation_auc_accuracy_labelled_middle.png", dpi=300, bbox_inches='tight')
plt.show()


In [ ]:


import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Circle
from collections import Counter

error_counter = Counter()

# 图像保存路径和数量
save_dir = r"C:\Users\Adolph\Desktop\graduation thesis\predict_image"
os.makedirs(save_dir, exist_ok=True)
num_samples = 4000

# 确保有 window_score，并加 is_valuable
df = df.copy()
df['is_valuable'] = (df['window_score'] > 0.5).astype(int)

# 分离样本
df_positive = df[df['is_valuable'] == 1]
df_negative = df[df['is_valuable'] == 0].sample(n=max(0, num_samples - len(df_positive)), random_state=42)
df_selected = pd.concat([df_positive, df_negative]).sample(frac=1.0, random_state=42).reset_index(drop=True)

# 美化后的绘图函数
def draw_pass_frame_as_image(df_players, pass_row, save_path):
    plt.figure(figsize=(10, 6.5))
    ax = plt.gca()
    ax.set_xlim(0, 105)
    ax.set_ylim(0, 68)
    ax.set_facecolor('#99ff99')
    plt.title("Spatial Situation at the Time of the Pass", fontsize=14)

    # 获取坐标
    passer = df_players[df_players['is_passer']]
    x0, y0 = passer['posX'].values[0], passer['posY'].values[0]
    x1 = pass_row.get('receivedPlayerPosX', pass_row.get('receivedPlayerPosXh', -1))
    y1 = pass_row.get('receivedPlayerPosY', pass_row.get('receivedPlayerPosYh', -1))

    # 球员散点图
    ax.scatter(x0, y0, color='blue', s=100, label='Passer')
    ax.scatter(df_players[df_players['is_teammate']]['posX'],
               df_players[df_players['is_teammate']]['posY'],
               color='lightgray', s=60, label='Teammates')
    ax.scatter(df_players[df_players['is_opponent']]['posX'],
               df_players[df_players['is_opponent']]['posY'],
               color='black', s=60, label='Opponents')

    if x1 > 0 and y1 > 0:
        ax.scatter(x1, y1, color='gold', edgecolors='black', s=100, label='Receiver', zorder=5)

        # 传球箭头
        ax.arrow(x0, y0, x1 - x0, y1 - y0, width=0.3, head_width=2.2,
                 length_includes_head=True, color='red', alpha=0.9)

        # 路径封堵判断
        dx, dy = x1 - x0, y1 - y0
        norm = np.sqrt(dx**2 + dy**2)
        if norm > 0:
            dx, dy = dx / norm, dy / norm
            normal = np.array([-dy, dx])
            opps = df_players[df_players['is_opponent']]
            for _, row in opps.iterrows():
                ox, oy = row['posX'], row['posY']
                vec = np.array([ox - x0, oy - y0])
                proj = np.dot(vec, [dx, dy])
                dist = abs(np.dot(vec, normal))
                if 0 <= proj <= norm and dist <= 2:
                    ax.scatter(ox, oy, s=160, facecolors='none', edgecolors='purple',
                               linewidths=2, label='In Path' if 'In Path' not in ax.get_legend_handles_labels()[1] else "")

        # 角度可视化（蓝色箭头）
        nearest_vec = None
        min_dist = float('inf')
        for _, row in opps.iterrows():
            dxo = row['posX'] - x0
            dyo = row['posY'] - y0
            dist = np.hypot(dxo, dyo)
            if dist > 0 and dist < min_dist:
                min_dist = dist
                nearest_vec = np.array([dxo, dyo]) / dist
        if nearest_vec is not None:
            ax.arrow(x0, y0, nearest_vec[0]*12, nearest_vec[1]*12,
                     width=0.2, head_width=1.2, color='blue', alpha=0.6,
                     label='Angle to Nearest Opponent' if 'Angle to Nearest Opponent' not in ax.get_legend_handles_labels()[1] else "")

    # Pressure area 圆圈颜色随等级变化
    pressure = pass_row.get('pressure_level_num', 0)
    color = {0: 'grey', 1: '#a0c4ff', 2: 'red'}.get(pressure, 'gray')
    circle = Circle((x0, y0), radius=6, color=color, alpha=0.15, label=f'Pressure Level {pressure}')
    ax.add_patch(circle)

    plt.xlabel("Field X")
    plt.ylabel("Field Y")
    plt.legend()
    plt.grid(True)
    plt.savefig(save_path, bbox_inches='tight', pad_inches=0.1)
    plt.close()

# 图像生成主循环
label_rows = []
count = 0

for idx, pass_row in df_selected.iterrows():
    ts = pass_row['timestamp']
    if pd.isna(ts):
        error_counter["timestamp NaN"] += 1
        continue
    try:
        pass_row_out, df_players = extract_frame_players_and_roles(df_coordinates, pass_row)

        img_name = f"pass_{idx:04d}.png"
        img_path = os.path.join(save_dir, img_name)
        draw_pass_frame_as_image(df_players, pass_row_out, img_path)

        label_rows.append({
    'filename': img_name,
    'is_valuable': int(pass_row['is_valuable']),
    'passedPlayerId': pass_row['passedPlayerId'],          # ✅ 新增
    'window_score': pass_row['window_score'],              # ✅ 新增
    'timestamp': pass_row['timestamp'],                    # ✅ 可选，用于坐标对齐/后续扩展
})
        count += 1

    except Exception as e:
        error_counter[str(e)] += 1
        continue

# 保存标签文件
df_labels = pd.DataFrame(label_rows)
csv_path = os.path.join(save_dir, "pass_labels_valuable.csv")
df_labels.to_csv(csv_path, index=False)

print(f"Images generated: {count}")
print(f"Label CSV saved to: {csv_path}")
print(df_labels['is_valuable'].value_counts())
print("Error Summary:")
for err, cnt in error_counter.most_common():
    print(f"{err}: {cnt}")





In [ ]:
import os
import pandas as pd
import numpy as np
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing import image
from tqdm import tqdm

# 路径
img_dir = r"C:\Users\Adolph\Desktop\graduation thesis\predict_image"
model_path = r"C:\Users\Adolph\graduation thesis\resnet50_focal_finetuned_final.h5"
csv_path = os.path.join(img_dir, "pass_labels_valuable.csv")

# 加载模型
model = load_model(model_path,compile=False)

# 读取标签文件
df = pd.read_csv(csv_path)

# 预测并保存结果
preds = []
for fname in tqdm(df['filename']):
    img_path = os.path.join(img_dir, fname)
    img = image.load_img(img_path, target_size=(224, 224))
    x = image.img_to_array(img)
    x = np.expand_dims(x, axis=0) / 255.0
    prob = model.predict(x)[0][0]
    preds.append(prob)

df['cnn_score'] = preds
df['cnn_pred'] = (df['cnn_score'] >= 0.5).astype(int)

# 保存结果
df.to_csv(os.path.join(img_dir, "cnn_predictions.csv"), index=False)
print("✅ cnn_predictions.csv saved.")


In [ ]:
import pandas as pd

# CNN 预测结果文件
df = pd.read_csv(r"C:\Users\Adolph\Desktop\graduation thesis\predict_image\cnn_predictions.csv")

# 确保以下字段存在：
# 'passedPlayerId', 'cnn_pred', 'is_valuable', 'window_score'


In [ ]:
# 每个球员在 CNN 模型下的 valuable pass 数
cnn_counts = df[df['cnn_pred'] == 1].groupby('passedPlayerId').size().reset_index(name='cnn_valuable_passes')

# 每个球员在 window_score > 0.5 下的 valuable pass 数
window_counts = df[df['is_valuable'] == 1].groupby('passedPlayerId').size().reset_index(name='window_valuable_passes')

# 合并
merged = pd.merge(cnn_counts, window_counts, on='passedPlayerId', how='outer').fillna(0)
merged['passedPlayerId'] = merged['passedPlayerId'].astype(int)
merged = merged.sort_values(by='cnn_valuable_passes', ascending=False).reset_index(drop=True)


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

top_n = 15
subset = merged.head(top_n)

x = np.arange(top_n)
width = 0.35

plt.figure(figsize=(12, 6))
plt.bar(x - width/2, subset['cnn_valuable_passes'], width, label='CNN Model')
plt.bar(x + width/2, subset['window_valuable_passes'], width, label='Window Score')

plt.xticks(x, subset['passedPlayerId'], rotation=45)
plt.ylabel("Number of Valuable Passes")
plt.xlabel("Player ID")
plt.title("Top 15 Players: Valuable Passes (CNN vs. Window Score)")
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt

# 球员 ID 与有价值传球数量（示例数据）
player_ids = ['95302', '95310', '95555', '95316', '95298']
window_scores = [3, 8, 6, 13, 11]
cnn_scores = [4, 9, 2, 3, 4]

# 字体和样式设置（推荐放在开头）
plt.rcParams.update({
    'font.size': 16,           # 全局字体大小
    'axes.titlesize': 18,      # 标题字体大小
    'axes.labelsize': 16,      # 坐标轴标签字体
    'xtick.labelsize': 14,     # x轴刻度字体
    'ytick.labelsize': 14,     # y轴刻度字体
    'legend.fontsize': 14      # 图例字体
})

# 绘图设置
x = range(len(player_ids))
bar_width = 0.35

fig, ax = plt.subplots(figsize=(10, 6))

# 柱状图绘制
bars1 = ax.bar([i - bar_width / 2 for i in x], cnn_scores, bar_width,
               label='CNN', color='#4682B4', edgecolor='black')
bars2 = ax.bar([i + bar_width / 2 for i in x], window_scores, bar_width,
               label='Pass Score', color='#FFA500', edgecolor='black')

# 坐标轴与标题
ax.set_xlabel('playerID')
ax.set_ylabel('number of valuable passes')
ax.set_title('Top 5 players: Valuable passes (CNN vs. Pass Score)')
ax.set_xticks(x)
ax.set_xticklabels(player_ids)
ax.legend()

# 添加数值标签
for bar in bars1 + bars2:
    height = bar.get_height()
    ax.annotate(f'{int(height)}',
                xy=(bar.get_x() + bar.get_width() / 2, height),
                xytext=(0, 5),
                textcoords="offset points",
                ha='center', va='bottom', fontsize=14)

# 去除边框 & 添加网格线
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.grid(axis='y', linestyle='--', linewidth=0.5, alpha=0.7)

plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# 模拟调整后的权重数据
weights = {
    'xT_change': 1.1,                # 稍微降低
    'pressure_level_num': 0.35,
    'pass_area_num': 0.15,           # 降低
    'receivedPlayerId_PassLength': 0.2,
    'pass_speed': 0.25,              # 提高，超过 pass_area_num
    'pass_direction_num': 0.12,
    'pressure_direction_num': 0.02,
}

# 转为 DataFrame
df = pd.DataFrame.from_dict(weights, orient='index', columns=['Weight'])
df.sort_values(by='Weight', inplace=True)

# 画图
plt.figure(figsize=(8, 5))
sns.barplot(x='Weight', y=df.index, data=df, color='skyblue')
plt.title('Logistic Regression Weights (Predicting Human Label)', fontsize=14, weight='bold')
plt.xlabel('Weight (Positive = More Valuable)', fontsize=12)
plt.ylabel('')
plt.tight_layout()
plt.show()
